# Step 3: LoRA-train MiniCPM5-2B as a Jev-style decision model

Trains a small LoRA adapter so MiniCPM5-2B answers each decision with **one option letter**, in exactly the prompt format the evaluation notebook scores: same chat template, same `Answer:` prefill, same letters. The loss is on that single answer token only.

At the end it fits a **temperature** on the validation set (so "90% sure" means about 90% right) and saves the adapter plus `calibration.json`.

### Run it on Kaggle
1. In the **step 2** notebook: **Save Version → Save & Run All**, so its `lora_data/` output exists.
2. In this notebook: right-hand panel **Add Input → Your Work → (step 2 notebook)**. Its `train.jsonl` / `val.jsonl` are found automatically.
3. **Settings → Accelerator → GPU T4 ×2**, **Internet → On**.
4. First run with `SMOKE = True` (≈5–10 min) to catch errors. Then set `SMOKE = False` and **Save Version → Save & Run All** for the real run (≈1.5–2.5 h). The adapter lands in the version's **Output**.

In [ ]:
!pip install -q -U "transformers>=5.6" peft accelerate
!pip uninstall -y -q torchao   # Kaggle's torchao 0.10 is too old for current peft, and we don't use it
# If an older transformers was already loaded in this session, drop it so the upgraded one is imported.
import sys
for _m in [m for m in sys.modules if m.split(".")[0] in ("transformers", "peft", "accelerate", "torchao")]:
    del sys.modules[_m]
import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)
# Still seeing "ImportError: cannot import name ..."? Use Run -> Restart & clear cell outputs, then Run All.

In [ ]:
BASE_MODEL = "openbmb/MiniCPM5-2B"
RUN_NAME = "minicpm5-2b-router"
SMOKE = True                     # True: tiny run to check everything works; False: the real training run

EPOCHS = 1
LR = 2e-4
WARMUP = 0.03                    # share of steps with a linear warm-up
TOKENS_PER_BATCH = 4096          # micro-batch size in (padded) tokens
GRAD_ACCUM = 4                   # micro-batches per optimizer step (~16k tokens per step)
MAX_TOKENS = 1536                # longer prompts are skipped
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
EVAL_EVERY = 150                 # optimizer steps between validation checks (also saves the adapter)
EVAL_ROWS = 600                  # validation rows used during training (all are used at the end)
SEED = 42
OUT_DIR = f"/kaggle/working/{RUN_NAME}"
PREFILL = "Answer:"              # must match the evaluation notebook
LETTERS = [chr(ord("A") + i) for i in range(26)]

if SMOKE:
    EVAL_EVERY, EVAL_ROWS = 20, 100

In [ ]:
import os, glob, json, math, time, random, collections
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
assert torch.cuda.is_available(), "Turn on a GPU: Settings -> Accelerator -> GPU T4 x2"
DEVICE = "cuda:0"
BF16 = torch.cuda.get_device_capability(0)[0] >= 8
DTYPE = torch.bfloat16 if BF16 else torch.float16
print(torch.cuda.get_device_name(0), "| compute dtype", DTYPE)


def find_data(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)) + \
           sorted(glob.glob(f"/kaggle/working/**/lora_data/{name}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{name} not found: attach the step 2 notebook's output (Add Input -> Your Work)")
    return hits[0]


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


train_rows = read_jsonl(find_data("train.jsonl"))
val_rows = read_jsonl(find_data("val.jsonl"))
if SMOKE:
    random.shuffle(train_rows)
    train_rows, val_rows = train_rows[:400], val_rows[:100]
print(len(train_rows), "train rows,", len(val_rows), "val rows from", os.path.dirname(find_data("train.jsonl")))
print(collections.Counter(r["variant"] for r in train_rows))

## Prompts and answer tokens (identical to the evaluation notebook)

In [ ]:
def build_prompt(tok, system, user):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    kwargs = dict(tokenize=False, add_generation_prompt=True, enable_thinking=False)
    try:
        text = tok.apply_chat_template(msgs, **kwargs)
    except Exception:  # template without a system role
        text = tok.apply_chat_template([{"role": "user", "content": system + "\n\n" + user}], **kwargs)
    return text + PREFILL


def letter_token_ids(tok, prompt):
    # The token id each letter gets right after the prefill ("Answer: A" or "Answer:A").
    base = tok(prompt, add_special_tokens=False)["input_ids"]
    for joiner in (" ", ""):
        ids = []
        for L in LETTERS:
            full = tok(prompt + joiner + L, add_special_tokens=False)["input_ids"]
            if len(full) != len(base) + 1 or full[:len(base)] != base:
                break
            ids.append(full[-1])
        if len(ids) == len(LETTERS) and len(set(ids)) == len(LETTERS):
            return joiner, ids
    raise ValueError("Option letters are not single, distinct tokens after the prefill")


def encode(tok, rows, letter_ids):
    out, skipped = [], 0
    for r in rows:
        ids = tok(build_prompt(tok, r["system"], r["user"]), add_special_tokens=False)["input_ids"]
        if len(ids) > MAX_TOKENS:
            skipped += 1
            continue
        out.append({"ids": ids, "target": letter_ids[r["answer_index"]], "answer": r["answer_index"],
                    "n_options": r["n_options"], "variant": r["variant"], "style": r["style"]})
    return out, skipped


def make_batches(items, shuffle=True):
    # Group similar lengths so little compute is spent on padding; cap each micro-batch at TOKENS_PER_BATCH.
    order = list(range(len(items)))
    if shuffle:
        random.shuffle(order)
    batches = []
    for start in range(0, len(order), 512):
        chunk = sorted(order[start:start + 512], key=lambda i: len(items[i]["ids"]))
        cur = []
        for i in chunk:
            longest = max([len(items[j]["ids"]) for j in cur] + [len(items[i]["ids"])])
            if cur and longest * (len(cur) + 1) > TOKENS_PER_BATCH:
                batches.append(cur)
                cur = []
            cur.append(i)
        if cur:
            batches.append(cur)
    if shuffle:
        random.shuffle(batches)
    return batches

## Load MiniCPM5-2B and attach LoRA

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
PAD_ID = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
JOINER, LETTER_IDS = letter_token_ids(tok, build_prompt(tok, train_rows[0]["system"], train_rows[0]["user"]))
print(f"letters follow the prefill with joiner {JOINER!r}")
print(build_prompt(tok, train_rows[0]["system"], train_rows[0]["user"])[-300:], "->", JOINER + LETTERS[train_rows[0]["answer_index"]])

train_items, sk1 = encode(tok, train_rows, LETTER_IDS)
val_items, sk2 = encode(tok, val_rows, LETTER_IDS)
lengths = [len(x["ids"]) for x in train_items]
print(f"{len(train_items)} train / {len(val_items)} val encoded; skipped {sk1 + sk2} over {MAX_TOKENS} tokens; "
      f"prompt tokens median {int(np.median(lengths))}, max {max(lengths)}, total {sum(lengths):,}")

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=DTYPE, device_map={"": 0})
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                         target_modules=TARGET_MODULES, task_type="CAUSAL_LM"))
for p in model.parameters():                      # train the adapter in fp32 for stable fp16 mixed precision
    if p.requires_grad:
        p.data = p.data.float()
model.print_trainable_parameters()

In [ ]:
def last_token_logits(model, batch_items):
    # Logits for the token right after the prompt (the answer letter), for a right-padded batch.
    n = max(len(x["ids"]) for x in batch_items)
    ids = torch.full((len(batch_items), n), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(batch_items), n), dtype=torch.long)
    for i, x in enumerate(batch_items):
        ids[i, :len(x["ids"])] = torch.tensor(x["ids"])
        mask[i, :len(x["ids"])] = 1
    ids, mask = ids.to(DEVICE), mask.to(DEVICE)
    core = model.get_base_model()                 # the CausalLM; LoRA layers are inside it
    hidden = core.model(input_ids=ids, attention_mask=mask).last_hidden_state
    last = hidden[torch.arange(len(batch_items), device=DEVICE), mask.sum(1) - 1]
    return core.lm_head(last).float()


@torch.no_grad()
def evaluate(items, max_rows=None):
    model.eval()
    sub = items[:max_rows] if max_rows else items
    rows, letter = [], torch.tensor(LETTER_IDS, device=DEVICE)
    for b in make_batches(sub, shuffle=False):
        batch = [sub[i] for i in b]
        with torch.autocast("cuda", dtype=DTYPE):
            logits = last_token_logits(model, batch)
        for x, lg in zip(batch, logits):
            opt = lg[letter[:x["n_options"]]]
            rows.append({"logits": opt.cpu().numpy(), "answer": x["answer"], "variant": x["variant"], "style": x["style"],
                         "letter_mass": float(torch.softmax(lg, -1)[letter[:x["n_options"]]].sum())})
    model.train()
    acc = np.mean([int(np.argmax(r["logits"]) == r["answer"]) for r in rows])
    by = collections.defaultdict(list)
    for r in rows:
        by[f"{r['style']}/{r['variant']}"].append(int(np.argmax(r["logits"]) == r["answer"]))
    return acc, {k: float(np.mean(v)) for k, v in sorted(by.items())}, float(np.mean([r["letter_mass"] for r in rows])), rows

## Train

In [ ]:
trainable = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.0)
# Fix every epoch's batches up front so the step count is exact.
plan = []
for e in range(math.ceil(EPOCHS)):
    bs = make_batches(train_items)
    plan += bs[:int(len(bs) * min(1.0, EPOCHS - e))]
total_steps = max(1, len(plan) // GRAD_ACCUM)
plan = plan[:total_steps * GRAD_ACCUM]
batches_per_epoch = len(make_batches(train_items))
warm = max(1, int(total_steps * WARMUP))
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm))))
scaler = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))
print(f"{total_steps} optimizer steps ({batches_per_epoch} micro-batches per epoch, {GRAD_ACCUM} per step)")

base_acc, base_by, base_mass, _ = evaluate(val_items, EVAL_ROWS)
print(f"before training: val acc {base_acc:.3f}, letter mass {base_mass:.2f}")
log = [{"step": 0, "val_acc": base_acc, "by": base_by, "letter_mass": base_mass}]
best = -1.0
step, micro, t0, seen_tokens, run_loss = 0, 0, time.time(), 0, []
model.train()
for b in plan:
    batch = [train_items[i] for i in b]
    targets = torch.tensor([x["target"] for x in batch], device=DEVICE)
    with torch.autocast("cuda", dtype=DTYPE):
        logits = last_token_logits(model, batch)
    loss = F.cross_entropy(logits, targets)
    scaler.scale(loss / GRAD_ACCUM).backward()
    run_loss.append(loss.item())
    seen_tokens += sum(len(x["ids"]) for x in batch)
    micro += 1
    if micro % GRAD_ACCUM:
        continue
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(trainable, 1.0)
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
    step += 1
    if step % 10 == 0:
        el = time.time() - t0
        print(f"step {step}/{total_steps}  loss {np.mean(run_loss[-10 * GRAD_ACCUM:]):.4f}  lr {sched.get_last_lr()[0]:.2e}  "
              f"{seen_tokens / el:,.0f} tok/s  ETA {el / step * (total_steps - step) / 60:.0f} min", flush=True)
    if step % EVAL_EVERY == 0 or step == total_steps:
        acc, by, mass, _ = evaluate(val_items, EVAL_ROWS)
        log.append({"step": step, "val_acc": acc, "by": by, "letter_mass": mass, "train_loss": float(np.mean(run_loss[-50:]))})
        print(f"  val acc {acc:.3f} (start {base_acc:.3f})  letter mass {mass:.2f}  " +
              "  ".join(f"{k} {v:.2f}" for k, v in by.items()), flush=True)
        model.save_pretrained(os.path.join(OUT_DIR, "adapter_last"))
        if acc > best:
            best = acc
            model.save_pretrained(os.path.join(OUT_DIR, "adapter"))
            print(f"  saved best adapter (val acc {acc:.3f})", flush=True)
        json.dump(log, open(os.path.join(OUT_DIR, "train_log.json"), "w"), indent=1)
print(f"training done in {(time.time() - t0) / 60:.1f} min; best val acc {best:.3f}")

## Calibrate and save

In [ ]:
from peft import PeftModel
from scipy.optimize import minimize_scalar

# Reload the best adapter, score the whole validation set, and fit one temperature on it.
import gc
del model, opt, trainable, sched, scaler
gc.collect(); torch.cuda.empty_cache()
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=DTYPE, device_map={"": 0})
model = PeftModel.from_pretrained(base_model, os.path.join(OUT_DIR, "adapter"))
acc, by, mass, rows = evaluate(val_items)


def nll_at(T):
    total = 0.0
    for r in rows:
        z = r["logits"] / T
        z = z - z.max()
        total -= z[r["answer"]] - np.log(np.exp(z).sum())
    return total / len(rows)


def ece_at(T, bins=15):
    conf, correct = [], []
    for r in rows:
        p = np.exp(r["logits"] / T - (r["logits"] / T).max()); p /= p.sum()
        conf.append(p.max()); correct.append(int(np.argmax(p) == r["answer"]))
    conf, correct = np.array(conf), np.array(correct)
    idx = np.minimum((conf * bins).astype(int), bins - 1)
    return float(sum((idx == b).mean() * abs(correct[idx == b].mean() - conf[idx == b].mean())
                     for b in range(bins) if (idx == b).any()))


T = float(minimize_scalar(nll_at, bounds=(0.05, 20), method="bounded").x)
calib = {"base_model": BASE_MODEL, "temperature": T, "prefill": PREFILL, "joiner": JOINER,
         "val_accuracy": acc, "val_by_type": by, "letter_mass": mass,
         "val_ece_raw": ece_at(1.0), "val_ece_calibrated": ece_at(T), "val_rows": len(rows),
         "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "targets": TARGET_MODULES},
         "train": {"epochs": EPOCHS, "lr": LR, "rows": len(train_items), "smoke": SMOKE}}
json.dump(calib, open(os.path.join(OUT_DIR, "adapter", "calibration.json"), "w"), indent=1)
print(json.dumps(calib, indent=1))
print("\nAdapter saved to", os.path.join(OUT_DIR, "adapter"))